# Why 20 mm? — The Science of Edge Depth Correction

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/8cH9azbsFifZ/climbing-science/main?labpath=notebooks%2F07_edge_depth_science.ipynb)

## Introduction — Why Edge Depth Matters

Maximal finger force depends heavily on the depth of the edge you're
pulling on. A 10 mm crimp feels dramatically harder than a 20 mm edge —
not because your fingers got weaker, but because the mechanical leverage
changes.

**Amca et al. (2012)** showed that maximal finger force decreases roughly
linearly as edge depth shrinks, at a rate of about **2.5 % per millimetre**
relative to a 20 mm reference edge.

This has practical consequences:
- Hangboard tests on different edges are **not directly comparable**
  without correction.
- The 20 mm edge has become the **de facto standard** for finger
  strength testing (IRCRA, Lattice Training, StrengthClimbing).
- If you test on a non-standard edge, you need to *normalise* the
  result before comparing it to published benchmarks.

This notebook walks through the correction model implemented in
`climbing_science.edge_depth`.

In [ ]:
# Install climbing-science (needed on Binder/Colab)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'climbing-science'])

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from climbing_science.edge_depth import (
    CORRECTION_RATE,
    MAX_EDGE_MM,
    MIN_EDGE_MM,
    REFERENCE_EDGE_MM,
    convert_force,
    correction_factor,
    estimate_force_at_depth,
    normalize_to_reference,
)

## Your Data

Enter your own measurements here. Every calculation below uses these
values, so you can re-run the notebook with your numbers.

In [ ]:
# ── Edit these values ──────────────────────────────────────────────
MY_FORCE_KG = 45        # max force on your test edge (kg)
MY_EDGE_MM = 15          # depth of the edge you tested on (mm)
MY_BODYWEIGHT_KG = 72    # your body weight (kg)
# ──────────────────────────────────────────────────────────────────

## The Linear Correction Model

The model is elegantly simple. For an edge of depth $d$ (mm) relative
to the reference depth $d_{\text{ref}}$ (20 mm), the **correction factor**
is:

$$
f(d) = 1 + r \times (d_{\text{ref}} - d)
$$

where $r$ = 0.025 (i.e. 2.5 % per mm), based on Amca et al. (2012).

**Interpretation:**
- $d = 20$ mm → $f = 1.0$ (no correction needed — this *is* the reference)
- $d < 20$ mm → $f > 1.0$ (smaller edge → force is corrected *upward*)
- $d > 20$ mm → $f < 1.0$ (larger edge → force is corrected *downward*)

To normalise a measured force to 20 mm:

$$
F_{20} = F_{\text{measured}} \times f(d)
$$

The model constants are:

In [ ]:
print(f"Reference edge:   {REFERENCE_EDGE_MM} mm")
print(f"Correction rate:  {CORRECTION_RATE} per mm  ({CORRECTION_RATE*100:.1f} %/mm)")
print(f"Valid range:      {MIN_EDGE_MM} mm – {MAX_EDGE_MM} mm")

### Correction Factor vs. Edge Depth

Let's plot the correction factor across the valid range.

In [ ]:
edges = np.linspace(MIN_EDGE_MM, MAX_EDGE_MM, 200)
factors = [correction_factor(e) for e in edges]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(edges, factors, linewidth=2, color="#2563eb")
ax.axhline(1.0, color="grey", linestyle="--", linewidth=0.8, label="No correction (20 mm)")
ax.axvline(REFERENCE_EDGE_MM, color="grey", linestyle=":", linewidth=0.8)

# Mark the user's edge
user_cf = correction_factor(MY_EDGE_MM)
ax.plot(MY_EDGE_MM, user_cf, "o", color="#dc2626", markersize=10, zorder=5,
        label=f"Your edge ({MY_EDGE_MM} mm → {user_cf:.3f})")

ax.set_xlabel("Edge depth (mm)", fontsize=12)
ax.set_ylabel("Correction factor", fontsize=12)
ax.set_title("Edge Depth Correction Factor (Amca et al. 2012)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Correction Factor Table

Common edge depths on popular hangboards, with their correction factors
and the percentage adjustment.

In [ ]:
common_edges = [8, 10, 12, 14, 15, 18, 20, 23, 25, 30, 35]

print(f"{'Edge (mm)':>10}  {'Factor':>8}  {'Adjustment':>12}  {'Note'}")
print("─" * 52)
for e in common_edges:
    cf = correction_factor(e)
    pct = (cf - 1.0) * 100
    sign = "+" if pct >= 0 else ""
    note = " ← reference" if e == REFERENCE_EDGE_MM else ""
    print(f"{e:>10}  {cf:>8.3f}  {sign}{pct:>+10.1f} %{note}")

## Interactive: Convert Your Force

Using `normalize_to_reference()`, we translate your measured force
on a non-standard edge to its 20 mm equivalent. This is the number
you'd compare against published benchmarks (StrengthClimbing, Lattice).

In [ ]:
force_20mm = normalize_to_reference(MY_FORCE_KG, MY_EDGE_MM)

print(f"Measured:  {MY_FORCE_KG:.1f} kg on {MY_EDGE_MM} mm")
print(f"Corrected: {force_20mm:.1f} kg on {REFERENCE_EDGE_MM:.0f} mm  "
      f"(factor = {correction_factor(MY_EDGE_MM):.3f})")
print()
print(f"Relative strength: {force_20mm / MY_BODYWEIGHT_KG * 100:.1f} % "
      f"of body weight ({MY_BODYWEIGHT_KG} kg)")

## Estimating Force at Any Depth

Given your normalised 20 mm force, `estimate_force_at_depth()` predicts
what you *would* pull on a different edge — useful for planning which
edge to train on.

In [ ]:
target_edges = [10, 12, 14, 15, 18, 20, 23, 25, 30]

print(f"Your 20 mm reference force: {force_20mm:.1f} kg\n")
print(f"{'Target edge':>12}  {'Est. force':>11}  {'% BW':>7}")
print("─" * 36)
for t in target_edges:
    est = estimate_force_at_depth(force_20mm, t)
    pct_bw = est / MY_BODYWEIGHT_KG * 100
    marker = " ←" if t == MY_EDGE_MM else ""
    print(f"{t:>10} mm  {est:>9.1f} kg  {pct_bw:>6.1f}%{marker}")

### Visualised: Your Estimated Force Curve

In [ ]:
edges_plot = np.linspace(MIN_EDGE_MM, MAX_EDGE_MM, 200)
forces_plot = [estimate_force_at_depth(force_20mm, e) for e in edges_plot]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(edges_plot, forces_plot, linewidth=2, color="#2563eb",
        label="Estimated force")
ax.axhline(force_20mm, color="grey", linestyle="--", linewidth=0.8,
           label=f"20 mm reference ({force_20mm:.1f} kg)")

# Mark actual measurement
ax.plot(MY_EDGE_MM, MY_FORCE_KG, "o", color="#dc2626", markersize=10,
        zorder=5, label=f"Your measurement ({MY_EDGE_MM} mm / {MY_FORCE_KG} kg)")

# Mark 20mm equivalent
ax.plot(REFERENCE_EDGE_MM, force_20mm, "s", color="#16a34a", markersize=10,
        zorder=5, label=f"20 mm equivalent ({force_20mm:.1f} kg)")

ax.set_xlabel("Edge depth (mm)", fontsize=12)
ax.set_ylabel("Estimated force (kg)", fontsize=12)
ax.set_title("Your Estimated Force Across Edge Depths", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Limitations

The linear 2.5 %/mm model is a useful approximation, but it has
known boundaries:

| Condition | Issue |
|---|---|
| **Very small edges (< 8 mm)** | Force loss accelerates non-linearly. Pain and skin friction become dominant limiters, not just finger strength. |
| **Very large edges (> 35 mm)** | The correction saturates — pulling on a 40 mm jug is barely different from a 35 mm edge. |
| **Different grip types** | The 2.5 %/mm rate was measured for **half-crimp**. Open-hand and full-crimp grips may have different depth sensitivity (Amca et al. 2012). |
| **Individual variation** | Hand size, finger proportions, and training history affect the actual rate. Treat the model as a population average. |
| **Edge profile** | Rounded vs. sharp edges of the same depth are not equivalent. The model assumes a standard flat edge. |

> **Rule of thumb**: The model is most reliable in the **10–30 mm range**,
> which covers the vast majority of hangboard testing.

## Recommendation: When to Use Which Edge

| Purpose | Recommended Edge |
|---|---|
| **Benchmark testing** (MVC-7, CF test) | **20 mm** — the universal standard. Results are directly comparable to Lattice, StrengthClimbing, and published norms. |
| **Performance-specific testing** | **The edge you project on.** If your goal route has 12 mm crimps, test on 12 mm and correct to 20 mm for tracking. |
| **Max recruitment training** | **14–18 mm** — enough stimulus for adaptation without excessive skin/joint stress. |
| **Repeater / endurance training** | **18–23 mm** — moderate depth that allows sustained effort without premature skin failure. |
| **Rehab / beginners** | **23–30 mm** — joint-friendly, lower injury risk. |

Always normalise results to 20 mm when comparing across sessions or athletes. The `climbing_science.edge_depth` module makes this a one-liner.

## References

- **Amca, A. M., Vigouroux, L., Aritan, S., & Berton, E. (2012).**
  Effect of hold depth and grip technique on maximal finger forces
  in rock climbing. *Journal of Sports Sciences*, 30(7), 669–677.
  doi:[10.1080/02640414.2012.658928](https://doi.org/10.1080/02640414.2012.658928)

- **StrengthClimbing.** Finger strength analyzer & edge depth correction.
  [strengthclimbing.com](https://strengthclimbing.com)

- **Lattice Training.** Normative finger-strength data.
  [latticetraining.com](https://latticetraining.com)